In [2]:

import pandas as pd
import numpy as np

transactions = pd.read_csv("../data/transactions_train.csv")
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

In [3]:
def precision_at_k(recommended_items, actual_items, k=10):
    recommended_k = recommended_items[:k]
    hits = len(set(recommended_k) & set(actual_items))
    return hits / k

def recall_at_k(recommended_items, actual_items, k=10):
    recommended_k = recommended_items[:k]
    hits = len(set(recommended_k) & set(actual_items))
    if len(actual_items) == 0:
        return 0
    return hits / len(actual_items)

def ndcg_at_k(recommended_items, actual_items, k=10):
    recommended_k = recommended_items[:k]
    
    dcg = 0
    for i, item in enumerate(recommended_k):
        if item in actual_items:
            dcg += 1 / np.log2(i + 2)
    
    ideal_hits = min(len(actual_items), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    
    if idcg == 0:
        return 0
    
    return dcg / idcg

In [4]:
cutoff_date = transactions["t_dat"].max() - pd.Timedelta(days=14)

train = transactions[transactions["t_dat"] <= cutoff_date]
test = transactions[transactions["t_dat"] > cutoff_date]

popular_items = train["article_id"].value_counts().head(10).index.tolist()

test_customers = test["customer_id"].value_counts().head(100).index

precision_scores = []
recall_scores = []
ndcg_scores = []

for customer_id in test_customers:
    actual_items = test[test["customer_id"] == customer_id]["article_id"].tolist()
    
    precision_scores.append(precision_at_k(popular_items, actual_items, k=10))
    recall_scores.append(recall_at_k(popular_items, actual_items, k=10))
    ndcg_scores.append(ndcg_at_k(popular_items, actual_items, k=10))

print("Popularity baseline:")
print("Precision@10:", np.mean(precision_scores))
print("Recall@10:", np.mean(recall_scores))
print("NDCG@10:", np.mean(ndcg_scores))

Popularity baseline:
Precision@10: 0.011000000000000001
Recall@10: 0.002470032393703778
NDCG@10: 0.016039691271554496
